In [1]:
# Colab cell 1
!pip install biopython pandas tqdm openpyxl
!pip install -q condacolab
import condacolab
condacolab.install()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.8 MB/s eta 0:00:00
⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [1]:
# Colab cell 2
!conda install -c bioconda -c conda-forge ncbi-amrfinderplus blast prodigal -y
!amrfinder_update

Channels:
 - bioconda
 - conda-forge
Platform: linux-64
Solving environment: - \ | / done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - blast
    - ncbi-amrfinderplus
    - prodigal


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    blast-2.17.0               |       h66d330f_0        80.9 MB  bioconda
    c-ares-1.34.6              |       hb03c661_0         203 KB  conda-forge
    ca-certificates-2026.4.22  |       hbd8a1cb_0         128 KB  conda-forge
    certifi-2026.4.22          |     pyhd8ed1ab_0         132 KB  conda-forge
    conda-24.11.3              |  py311h38be061_0         1.1 MB  conda-forge
    curl-8.18.0                |       h4e3cde8_0         186 KB  conda-forge
    entrez-direct-24.0         |       he881be0_0        16.3 MB  bioconda
    gsl-2.7                    |       he838d99_0         3.2 MB  conda-forge
    

In [2]:
# Colab cell 4
import os, glob, subprocess, pandas as pd
from tqdm import tqdm

os.makedirs("amrfinder_results", exist_ok=True)

fasta_files = glob.glob("isolates/*")

for fasta in tqdm(fasta_files):
    sample = os.path.basename(fasta).split(".")[0]
    out_file = f"amrfinder_results/{sample}_amrfinder.tsv"

    cmd = [
        "amrfinder",
        "-n", fasta,
        "-o", out_file
    ]

    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"AMRFinder failed for {sample}: {e}")

print("Done.")

0it [00:00, ?it/s]

Done.


In [3]:
# Colab cell 5
import pandas as pd, glob, os

all_results = []

for file in glob.glob("amrfinder_results/*_amrfinder.tsv"):
    sample = os.path.basename(file).replace("_amrfinder.tsv", "")

    if os.path.getsize(file) == 0:
        continue

    df = pd.read_csv(file, sep="\t")
    df["isolate_id"] = sample
    all_results.append(df)

if all_results:
    amr_df = pd.concat(all_results, ignore_index=True)
else:
    amr_df = pd.DataFrame()

amr_df.head()

""


In [4]:
# Colab cell 6
print(amr_df.columns.tolist())

# Try to standardize common AMRFinderPlus columns
keep_cols = []

for col in [
    "isolate_id",
    "Gene symbol",
    "Sequence name",
    "Scope",
    "Element type",
    "Element subtype",
    "Class",
    "Subclass",
    "Method",
    "% Identity",
    "% Coverage of reference sequence",
    "Start",
    "Stop",
    "Strand"
]:
    if col in amr_df.columns:
        keep_cols.append(col)

amr_summary = amr_df[keep_cols].copy()
amr_summary.head()

[]


""


In [5]:
# Colab cell 7
priority_genes = [
    "blaKPC", "blaNDM", "blaOXA", "blaVIM", "blaIMP",
    "mcr", "mecA", "vanA", "vanB", "aac", "aph", "erm", "tet"
]

def is_priority_gene(gene):
    gene = str(gene).lower()
    return any(pg.lower() in gene for pg in priority_genes)

gene_col = "Gene symbol" if "Gene symbol" in amr_summary.columns else amr_summary.columns[1]

amr_summary["priority_marker"] = amr_summary[gene_col].apply(is_priority_gene)

amr_summary.sort_values(["isolate_id", "priority_marker"], ascending=[True, False]).head(20)

IndexError: index 1 is out of bounds for axis 0 with size 0

In [6]:
# SAFE Cell 7: Flag clinically important AMR genes

priority_genes = [
    "blaKPC", "blaNDM", "blaOXA", "blaVIM", "blaIMP",
    "mcr", "mecA", "vanA", "vanB", "aac", "aph", "erm", "tet"
]

if amr_summary.empty:
    print("⚠️ amr_summary is empty.")
    print("This means AMRFinderPlus did not return/load any AMR hits.")
    print("Check the previous AMRFinder result files first.")
else:
    print("Columns available:")
    print(amr_summary.columns.tolist())

    gene_col = "Gene symbol" if "Gene symbol" in amr_summary.columns else None

    if gene_col is None:
        possible_gene_cols = [
            c for c in amr_summary.columns
            if "gene" in c.lower() or "symbol" in c.lower() or "element" in c.lower()
        ]
        print("Possible gene columns:", possible_gene_cols)

        if possible_gene_cols:
            gene_col = possible_gene_cols[0]
        else:
            raise ValueError("No gene-like column found in AMR summary.")

    def is_priority_gene(gene):
        gene = str(gene).lower()
        return any(pg.lower() in gene for pg in priority_genes)

    amr_summary["priority_marker"] = amr_summary[gene_col].apply(is_priority_gene)

    display(
        amr_summary
        .sort_values(["isolate_id", "priority_marker"], ascending=[True, False])
        .head(20)
    )

⚠️ amr_summary is empty.
This means AMRFinderPlus did not return/load any AMR hits.
Check the previous AMRFinder result files first.


In [7]:
# DEBUG: check AMRFinder output files
import glob, os, pandas as pd

files = glob.glob("amrfinder_results/*_amrfinder.tsv")
print("Number of AMRFinder result files:", len(files))

for f in files[:10]:
    print(f, "size:", os.path.getsize(f))

    try:
        temp = pd.read_csv(f, sep="\t")
        print("rows:", len(temp))
        print("columns:", temp.columns.tolist())
        display(temp.head())
    except Exception as e:
        print("Could not read:", e)

Number of AMRFinder result files: 0


In [8]:
# SAFE Cell 7: Flag clinically important AMR genes

priority_genes = [
    "blaKPC", "blaNDM", "blaOXA", "blaVIM", "blaIMP",
    "mcr", "mecA", "vanA", "vanB", "aac", "aph", "erm", "tet"
]

if amr_summary.empty:
    print("⚠️ amr_summary is empty.")
    print("This means AMRFinderPlus did not return/load any AMR hits.")
    print("Check the previous AMRFinder result files first.")
else:
    print("Columns available:")
    print(amr_summary.columns.tolist())

    gene_col = "Gene symbol" if "Gene symbol" in amr_summary.columns else None

    if gene_col is None:
        possible_gene_cols = [
            c for c in amr_summary.columns
            if "gene" in c.lower() or "symbol" in c.lower() or "element" in c.lower()
        ]
        print("Possible gene columns:", possible_gene_cols)

        if possible_gene_cols:
            gene_col = possible_gene_cols[0]
        else:
            raise ValueError("No gene-like column found in AMR summary.")

    def is_priority_gene(gene):
        gene = str(gene).lower()
        return any(pg.lower() in gene for pg in priority_genes)

    amr_summary["priority_marker"] = amr_summary[gene_col].apply(is_priority_gene)

    display(
        amr_summary
        .sort_values(["isolate_id", "priority_marker"], ascending=[True, False])
        .head(20)
    )

⚠️ amr_summary is empty.
This means AMRFinderPlus did not return/load any AMR hits.
Check the previous AMRFinder result files first.


In [9]:
import os, glob

print("Files in isolates folder:")
for f in glob.glob("isolates/*"):
    print(f, "size:", os.path.getsize(f))

Files in isolates folder:


In [10]:
!amrfinder -n isolates/YOUR_FILE_NAME.fasta -o test_amrfinder.tsv
!cat test_amrfinder.tsv

Running: amrfinder -n isolates/YOUR_FILE_NAME.fasta -o test_amrfinder.tsv
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
*** ERROR ***
No valid AMRFinder database is found.
This directory (or symbolic link to directory) is not found: /usr/local/share/amrfinderplus/data/latest
To download the latest version to the default directory run: amrfinder -u
cat: test_amrfinder.tsv: No such file or directory


In [11]:
!amrfinder -u

Running: amrfinder -u
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
Running: /usr/local/bin/amrfinder_update -d /usr/local/share/amrfinderplus/data
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /usr/local/share/amrfinderplus/data/2026-03-24.1/
Running: /usr/local/bin/amrfinder_index /usr/local/share/amrfinderplus/data/2026-03-24.1/
Indexing
amrfinder_index took 5 seconds to complete
amrfinder_update took 17 seconds to complete
Database directory: /usr/local/share/amrfinderplus/data/2026-03-24.1
Database version: 2026-03-24.1
amrfinder took 17 seconds to complete


In [13]:
!amrfinder_update

Running: amrfinder_update
Looking up the published databases at https://ftp.ncbi.nlm.nih.gov/pathogen/Antimicrobial_resistance/AMRFinderPlus/database/
Looking for the target directory: /usr/local/bin/data/2026-03-24.1/
Skipping update
Use amrfinder --force_update to overwrite the existing database
amrfinder_update took 0 seconds to complete


In [14]:
!amrfinder --version
!ls /usr/local/share/amrfinderplus/data/latest

4.2.7
AMR_CDS.fa
AMR_CDS.fa.ndb
AMR_CDS.fa.nhr
AMR_CDS.fa.nin
AMR_CDS.fa.njs
AMR_CDS.fa.not
AMR_CDS.fa.nsq
AMR_CDS.fa.ntf
AMR_CDS.fa.nto
AMR_DNA-Acinetobacter_baumannii.fa
AMR_DNA-Acinetobacter_baumannii.fa.ndb
AMR_DNA-Acinetobacter_baumannii.fa.nhr
AMR_DNA-Acinetobacter_baumannii.fa.nin
AMR_DNA-Acinetobacter_baumannii.fa.njs
AMR_DNA-Acinetobacter_baumannii.fa.not
AMR_DNA-Acinetobacter_baumannii.fa.nsq
AMR_DNA-Acinetobacter_baumannii.fa.ntf
AMR_DNA-Acinetobacter_baumannii.fa.nto
AMR_DNA-Acinetobacter_baumannii.tsv
AMR_DNA-Bordetella_pertussis.fa
AMR_DNA-Bordetella_pertussis.fa.ndb
AMR_DNA-Bordetella_pertussis.fa.nhr
AMR_DNA-Bordetella_pertussis.fa.nin
AMR_DNA-Bordetella_pertussis.fa.njs
AMR_DNA-Bordetella_pertussis.fa.not
AMR_DNA-Bordetella_pertussis.fa.nsq
AMR_DNA-Bordetella_pertussis.fa.ntf
AMR_DNA-Bordetella_pertussis.fa.nto
AMR_DNA-Bordetella_pertussis.tsv
AMR_DNA-Campylobacter.fa
AMR_DNA-Campylobacter.fa.ndb
AMR_DNA-Campylobacter.fa.nhr
AMR_DNA-Campylobacter.fa.nin
AMR_DNA-Campylo

In [15]:
!amrfinder -n isolates/YOUR_FILE_NAME.fasta -o test_amrfinder.tsv
!cat test_amrfinder.tsv | head

Running: amrfinder -n isolates/YOUR_FILE_NAME.fasta -o test_amrfinder.tsv
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
Database directory: /usr/local/share/amrfinderplus/data/2026-03-24.1
Database version: 2026-03-24.1
AMRFinder translated nucleotide search
  - include -O ORGANISM, --organism ORGANISM option to add mutation searches and suppress common proteins
*** ERROR ***
Cannot open file 'isolates/YOUR_FILE_NAME.fasta' to get file size
cat: test_amrfinder.tsv: No such file or directory


In [16]:
import glob, os

print(glob.glob("isolates/*"))

[]


In [17]:
from google.colab import files
import os, shutil

os.makedirs("isolates", exist_ok=True)

uploaded = files.upload()

for fn in uploaded.keys():
    shutil.move(fn, f"isolates/{fn}")

print(glob.glob("isolates/*"))

[]


In [18]:
!wget -O klebsiella.fna https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/240/185/GCF_000240185.1_ASM24018v1/GCF_000240185.1_ASM24018v1_genomic.fna

--2026-04-28 19:00:56--  https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/240/185/GCF_000240185.1_ASM24018v1/GCF_000240185.1_ASM24018v1_genomic.fna
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.10, 2607:f220:41e:250::13, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-28 19:00:57 ERROR 404: Not Found.



In [19]:
import os
os.makedirs("isolates", exist_ok=True)
!mv klebsiella.fna isolates/

In [20]:
!amrfinder -n isolates/klebsiella.fna -o test_amrfinder.tsv
!cat test_amrfinder.tsv | head -30

Running: amrfinder -n isolates/klebsiella.fna -o test_amrfinder.tsv
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
Database directory: /usr/local/share/amrfinderplus/data/2026-03-24.1
Database version: 2026-03-24.1
AMRFinder translated nucleotide search
  - include -O ORGANISM, --organism ORGANISM option to add mutation searches and suppress common proteins
Making report
amrfinder took 0 seconds to complete
Protein id	Contig id	Start	Stop	Strand	Element symbol	Element name	Scope	Type	Subtype	Class	Subclass	Method	Target length	Reference sequence length	% Coverage of reference	% Identity to reference	Alignment length	Closest reference accession	Closest reference name	HMM accession	HMM description


In [21]:
# Remove empty file
!rm -f isolates/klebsiella.fna test_amrfinder.tsv

# Check folder
!ls -lh isolates

total 0


In [22]:
# Find AMRFinder test FASTA files available in Colab
!find /usr/local -name "*.fa" | head -20
!find /usr/local -name "*.fna" | head -20

/usr/local/bin/data/2026-03-24.1/AMR_DNA-Salmonella.fa
/usr/local/bin/data/2026-03-24.1/AMR_CDS.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Enterococcus_faecium.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Enterococcus_faecalis.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Neisseria_gonorrhoeae.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Clostridioides_difficile.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Escherichia.fa
/usr/local/bin/data/2026-03-24.1/AMRProt.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Bordetella_pertussis.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Staphylococcus_aureus.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Streptococcus_pneumoniae.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Acinetobacter_baumannii.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Campylobacter.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Klebsiella_pneumoniae.fa
/usr/local/bin/data/2026-03-24.1/AMR_DNA-Klebsiella_oxytoca.fa
/usr/local/bin/data/2026-03-24.1/AMRProt-susceptible.fa
/usr/local/share/easel/d

In [24]:
# Use AMRFinder's own test DNA if available
!find /usr/local -name "test_dna.fa"

In [25]:
!rm -rf isolates
!mkdir isolates

!wget -O isolates/test_dna.fa https://raw.githubusercontent.com/ncbi/amr/master/test_dna.fa

!ls -lh isolates/test_dna.fa
!head isolates/test_dna.fa

--2026-04-28 19:07:33--  https://raw.githubusercontent.com/ncbi/amr/master/test_dna.fa
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32205 (31K) [text/plain]
Saving to: ‘isolates/test_dna.fa’

isolates/test_dna.f 100%[===================>]  31.45K  --.-KB/s    in 0.002s  

2026-04-28 19:07:33 (17.9 MB/s) - ‘isolates/test_dna.fa’ saved [32205/32205]

-rw-r--r-- 1 root root 32K Apr 28 19:07 isolates/test_dna.fa
>contig01 blaTEM-156_cds
AACCCCTATTTGTTTATTTTTCTAAATACATTCAAATATGTATCCGCTCATGATACAATAACCCTGATAA
ATGCTTCAATAATATTGAAAAAGGAAGAGTATGAGTATTCAACATTTCCGTGTCGCCCTTATTCCCTTTT
TTGCGGCATTTTGCCTTCCTGTTTTTGCTCACCCAGAAACGCTGGTGAAAGTAAAAGATGCTGAAGATCA
GTTGGGTGCACGAGTGGGTTACATCGAACTGGATCTCAACAGCGGTAAGATCCTTGAGAGTTTTCGCCCC
GAAGAACGTTTTCCAATGATGAGCACTTTTAAAGTTCTGCTATG

In [26]:
!amrfinder -n isolates/test_dna.fa -O Escherichia --plus -o test_amrfinder.tsv
!cat test_amrfinder.tsv | head -30

Running: amrfinder -n isolates/test_dna.fa -O Escherichia --plus -o test_amrfinder.tsv
The number of threads cannot be greater than 2 on this computer
The current number of threads is 4, reducing to 2
Software directory: /usr/local/bin/
Software version: 4.2.7
Reverting to hard coded directory: /usr/local/share/amrfinderplus/data/latest
Database directory: /usr/local/share/amrfinderplus/data/2026-03-24.1
Database version: 2026-03-24.1
AMRFinder translated nucleotide and mutation search
Running blastx
Running blastn
Running stxtyper
Making report
amrfinder took 9 seconds to complete
Protein id	Contig id	Start	Stop	Strand	Element symbol	Element name	Scope	Type	Subtype	Class	Subclass	Method	Target length	Reference sequence length	% Coverage of reference	% Identity to reference	Alignment length	Closest reference accession	Closest reference name	HMM accession	HMM description
NA	contig01	1	984	+	blaTEMp_G162T	Escherichia amoxicillin-clavulanic acid/piperacillin-tazobactam/ticarcillin-clavula

In [27]:
import pandas as pd

# Load AMRFinder output
amr_df = pd.read_csv("test_amrfinder.tsv", sep="\t")

# Extract gene symbols
gene_col = "Element symbol"

detected_genes = amr_df[gene_col].dropna().unique().tolist()

print("Detected AMR genes:")
for g in detected_genes:
    print(g)

Detected AMR genes:
blaTEMp_G162T
blaTEM-156
blaPDC
blaOXA
vanG
blaEC
blaTEM
aph(3'')-Ib
sul2
qacR
emrD3
pmrB_C84R
23S_A2058T
nfsA_K141Ter
nfsA_R15C
ampC_T-14TGT
stxA2
stx2a_operon
stxB2


In [28]:
target_genes = ["blaKPC", "blaNDM", "mcr", "mecA", "blaOXA", "blaTEM"]

In [29]:
def match_target_gene(gene):
    gene = gene.lower()
    return any(t.lower() in gene for t in target_genes)

amr_df["crispr_target"] = amr_df[gene_col].apply(match_target_gene)

amr_targets = amr_df[amr_df["crispr_target"] == True]

amr_targets[[gene_col, "Class", "Subclass"]]

,Element symbol,Class,Subclass
0,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...
1,blaTEM-156,BETA-LACTAM,BETA-LACTAM
3,blaOXA,BETA-LACTAM,BETA-LACTAM
6,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...
7,blaTEM,BETA-LACTAM,BETA-LACTAM
10,blaOXA,BETA-LACTAM,BETA-LACTAM
11,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...
12,blaTEM,BETA-LACTAM,BETA-LACTAM


In [30]:
# Normalize gene names (important)
amr_targets["clean_gene"] = amr_targets["Element symbol"].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()

amr_targets[["Element symbol", "clean_gene"]]

/tmp/ipykernel_36292/3621329773.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  amr_targets["clean_gene"] = amr_targets["Element symbol"].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()


,Element symbol,clean_gene
0,blaTEMp_G162T,blatempg162t
1,blaTEM-156,blatem156
3,blaOXA,blaoxa
6,blaTEMp_G162T,blatempg162t
7,blaTEM,blatem
10,blaOXA,blaoxa
11,blaTEMp_G162T,blatempg162t
12,blaTEM,blatem


In [31]:
from google.colab import files
uploaded = files.upload()

Saving cross_model_reference_ranked_guides.csv to cross_model_reference_ranked_guides.csv


In [32]:
import pandas as pd

crispr_df = pd.read_csv("cross_model_reference_ranked_guides.csv")

print(crispr_df.columns)
crispr_df.head()

Index(['gene', 'position', 'strand', 'spacer', 'pam', 'gc_content',
       'on_target_score', 'offtarget_hits_0mm', 'offtarget_hits_1mm',
       'offtarget_hits_2mm', 'offtarget_hits_3mm', 'offtarget_penalty',
       'specificity_score', 'total_strains', 'perfect_match_strains',
       'one_mismatch_or_better_strains', 'pam_supported_strains',
       'perfect_match_fraction', 'one_mismatch_or_better_fraction',
       'pam_supported_fraction', 'conservation_score', 'mean_strain_score',
       'min_strain_score', 'std_strain_score', 'final_score', 'classification',
       'guide_key', 'weight_scheme', 'final_score_new', 'classification_new',
       'rank_on_target', 'rank_off_target', 'rank_conservation', 'rank_final',
       'consensus_score', 'rank_consensus'],
      dtype='object')


,gene,position,strand,spacer,pam,gc_content,on_target_score,offtarget_hits_0mm,offtarget_hits_1mm,offtarget_hits_2mm,...,guide_key,weight_scheme,final_score_new,classification_new,rank_on_target,rank_off_target,rank_conservation,rank_final,consensus_score,rank_consensus
0,blaKPC,171,+,AGAGCCTTACTGCCCGAAGG,CGG,60.0,82.0,0,0,0,...,blaKPC|171|AGAGCCTTACTGCCCGAAGG,40/30/30,92.8,Excellent,3.0,1.0,1.0,3.0,94.0,3.0
1,blaKPC,174,+,GCCTTACTGCCCGAAGGCGG,CGG,70.0,82.0,0,0,0,...,blaKPC|174|GCCTTACTGCCCGAAGGCGG,40/30/30,92.8,Excellent,3.0,1.0,1.0,3.0,94.0,3.0
2,blaKPC,248,+,CGGCTCCATCGGTGTGTACG,CGG,65.0,82.0,0,0,0,...,blaKPC|248|CGGCTCCATCGGTGTGTACG,40/30/30,92.8,Excellent,3.0,1.0,1.0,3.0,94.0,3.0
3,blaKPC,282,-,AGCGACGGAATAGTGTATGG,CGG,50.0,82.0,0,0,0,...,blaKPC|282|AGCGACGGAATAGTGTATGG,40/30/30,92.8,Excellent,3.0,1.0,1.0,3.0,94.0,3.0
4,blaKPC,205,-,AGCGACGGAATAGTGTATGG,CGG,50.0,82.0,0,0,0,...,blaKPC|205|AGCGACGGAATAGTGTATGG,40/30/30,92.8,Excellent,3.0,1.0,1.0,3.0,94.0,3.0


In [33]:
crispr_df["clean_gene"] = crispr_df["gene"].str.replace(r'[^a-zA-Z0-9]', '', regex=True).str.lower()

In [34]:
final_df = amr_targets.merge(
    crispr_df,
    on="clean_gene",
    how="left"
)

final_df.head()

,Protein id,Contig id,Start,Stop,Strand,Element symbol,Element name,Scope,Type,Subtype,...,guide_key,weight_scheme,final_score_new,classification_new,rank_on_target,rank_off_target,rank_conservation,rank_final,consensus_score,rank_consensus
0,NaN,contig01,1,984,+,blaTEMp_G162T,Escherichia amoxicillin-clavulanic acid/pipera...,core,AMR,POINT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,contig01,101,958,+,blaTEM-156,class A beta-lactamase TEM-156,core,AMR,AMR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,contig03,101,802,+,blaOXA,OXA-48 family class D beta-lactamase,core,AMR,AMR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,contig08,1,700,+,blaTEMp_G162T,Escherichia amoxicillin-clavulanic acid/pipera...,core,AMR,POINT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,contig08,101,700,+,blaTEM,TEM family class A beta-lactamase,core,AMR,AMR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:
print("AMR genes:", amr_targets["clean_gene"].unique())
print("CRISPR genes:", crispr_df["clean_gene"].unique())

AMR genes: ['blatempg162t' 'blatem156' 'blaoxa' 'blatem']
CRISPR genes: ['blakpc' 'blandm1' 'mcr1' 'meca']


In [36]:
final_df = final_df.sort_values(
    ["clean_gene", "final_score"],
    ascending=[True, False]
)

final_df["rank"] = final_df.groupby("clean_gene").cumcount() + 1

top_guides = final_df[final_df["rank"] <= 5]

In [37]:
camda_output = top_guides[[
    "clean_gene",
    "Element symbol",
    "Class",
    "Subclass",
    "spacer",
    "pam",
    "final_score",
    "rank"
]]

camda_output.head(20)

,clean_gene,Element symbol,Class,Subclass,spacer,pam,final_score,rank
2,blaoxa,blaOXA,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
5,blaoxa,blaOXA,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,2
4,blatem,blaTEM,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
7,blatem,blaTEM,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,2
1,blatem156,blaTEM-156,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
0,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,1
3,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,2
6,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,3


In [38]:
camda_output.to_csv("CAMDA_AMR_CRISPR_output.csv", index=False)

from google.colab import files
files.download("CAMDA_AMR_CRISPR_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
"final_score"

'final_score'

In [40]:
import pandas as pd

df = pd.read_csv("CAMDA_AMR_CRISPR_output.csv")

print(df.shape)
df.head(10)

(8, 8)


,clean_gene,Element symbol,Class,Subclass,spacer,pam,final_score,rank
0,blaoxa,blaOXA,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
1,blaoxa,blaOXA,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,2
2,blatem,blaTEM,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
3,blatem,blaTEM,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,2
4,blatem156,blaTEM-156,BETA-LACTAM,BETA-LACTAM,NaN,NaN,NaN,1
5,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,1
6,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,2
7,blatempg162t,blaTEMp_G162T,BETA-LACTAM,AMOXICILLIN-CLAVULANIC_ACID/PIPERACILLIN-TAZOB...,NaN,NaN,NaN,3


In [41]:
print(df.columns)
print(df["clean_gene"].unique())

Index(['clean_gene', 'Element symbol', 'Class', 'Subclass', 'spacer', 'pam',
       'final_score', 'rank'],
      dtype='object')
['blaoxa' 'blatem' 'blatem156' 'blatempg162t']


In [42]:
print("AMR genes:")
print(amr_targets["clean_gene"].unique())

print("CRISPR genes:")
print(crispr_df["clean_gene"].unique())

AMR genes:
['blatempg162t' 'blatem156' 'blaoxa' 'blatem']
CRISPR genes:
['blakpc' 'blandm1' 'mcr1' 'meca']


In [43]:
gene_map = {
    "blaoxa": "blaoxa",
    "blatem": "blatem",
    "blatem156": "blatem",
    "blatempg162t": "blatem"
}

amr_targets["mapped_gene"] = amr_targets["clean_gene"].map(gene_map)

crispr_df["mapped_gene"] = crispr_df["clean_gene"]

final_df = amr_targets.merge(
    crispr_df,
    on="mapped_gene",
    how="left"
)

final_df[["Element symbol", "mapped_gene", "spacer", "pam", "final_score"]].head(20)

/tmp/ipykernel_36292/2761722838.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  amr_targets["mapped_gene"] = amr_targets["clean_gene"].map(gene_map)


,Element symbol,mapped_gene,spacer,pam,final_score
0,blaTEMp_G162T,blatem,NaN,NaN,NaN
1,blaTEM-156,blatem,NaN,NaN,NaN
2,blaOXA,blaoxa,NaN,NaN,NaN
3,blaTEMp_G162T,blatem,NaN,NaN,NaN
4,blaTEM,blatem,NaN,NaN,NaN
5,blaOXA,blaoxa,NaN,NaN,NaN
6,blaTEMp_G162T,blatem,NaN,NaN,NaN
7,blaTEM,blatem,NaN,NaN,NaN


In [44]:
from Bio import SeqIO
from Bio.Seq import Seq
import pandas as pd, re

records = list(SeqIO.parse("isolates/test_dna.fa", "fasta"))

def gc(seq):
    return round((seq.count("G") + seq.count("C")) / len(seq), 3)

guides = []

for _, row in amr_targets.iterrows():
    contig = row["Contig id"]
    start = int(row["Start"])
    stop = int(row["Stop"])
    gene = row["Element symbol"]

    rec = next(r for r in records if r.id == contig)
    seq = str(rec.seq).upper()[min(start, stop)-1:max(start, stop)]

    for i in range(len(seq)-23):
        spacer = seq[i:i+20]
        pam = seq[i+20:i+23]

        if re.match("^[ATGC]{20}$", spacer) and re.match("^[ATGC]GG$", pam):
            score = 1 - abs(gc(spacer)-0.5)
            guides.append({
                "gene": gene,
                "clean_gene": row["clean_gene"],
                "Class": row["Class"],
                "Subclass": row["Subclass"],
                "spacer": spacer,
                "pam": pam,
                "gc_content": gc(spacer),
                "final_score": round(score, 4)
            })

guide_df = pd.DataFrame(guides)

guide_df = guide_df.sort_values(
    ["clean_gene", "final_score"],
    ascending=[True, False]
)

guide_df["rank"] = guide_df.groupby("clean_gene").cumcount() + 1

camda_fixed = guide_df[guide_df["rank"] <= 5]

camda_fixed.to_csv("CAMDA_FINAL_FIXED_OUTPUT.csv", index=False)

camda_fixed.head(20)

,gene,clean_gene,Class,Subclass,spacer,pam,gc_content,final_score,rank
108,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,AGCCTTATCGGCTGTGTTGA,TGG,0.5,1.0,1
110,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,AACGATGATTGGCATGCCTG,CGG,0.5,1.0,2
116,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,CGAACATAAAACCCAAGGCG,TGG,0.5,1.0,3
118,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,CTGGAACGAGAATACACAGC,AGG,0.5,1.0,4
140,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,GACAGTTTTTGGCTCGATGG,TGG,0.5,1.0,5
185,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TTTTGCTCACCCAGAAACGC,TGG,0.5,1.0,1
190,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,ACGAGTGGGTTACATCGAAC,TGG,0.5,1.0,2
194,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,GTATTATCCCGTGTTGACGC,CGG,0.5,1.0,3
195,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TATTATCCCGTGTTGACGCC,GGG,0.5,1.0,4
325,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TTTTGCTCACCCAGAAACGC,TGG,0.5,1.0,5


In [45]:
from google.colab import files
files.download("CAMDA_FINAL_FIXED_OUTPUT.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [46]:
import pandas as pd

df = pd.read_csv("CAMDA_FINAL_FIXED_OUTPUT.csv")

print(df.shape)
df.head(10)

(20, 9)


,gene,clean_gene,Class,Subclass,spacer,pam,gc_content,final_score,rank
0,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,AGCCTTATCGGCTGTGTTGA,TGG,0.5,1.0,1
1,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,AACGATGATTGGCATGCCTG,CGG,0.5,1.0,2
2,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,CGAACATAAAACCCAAGGCG,TGG,0.5,1.0,3
3,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,CTGGAACGAGAATACACAGC,AGG,0.5,1.0,4
4,blaOXA,blaoxa,BETA-LACTAM,BETA-LACTAM,GACAGTTTTTGGCTCGATGG,TGG,0.5,1.0,5
5,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TTTTGCTCACCCAGAAACGC,TGG,0.5,1.0,1
6,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,ACGAGTGGGTTACATCGAAC,TGG,0.5,1.0,2
7,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,GTATTATCCCGTGTTGACGC,CGG,0.5,1.0,3
8,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TATTATCCCGTGTTGACGCC,GGG,0.5,1.0,4
9,blaTEM,blatem,BETA-LACTAM,BETA-LACTAM,TTTTGCTCACCCAGAAACGC,TGG,0.5,1.0,5


In [47]:
df["isolate_id"] = "test_isolate_1"

df = df[[
    "isolate_id",
    "clean_gene",
    "gene",
    "Class",
    "Subclass",
    "spacer",
    "pam",
    "final_score",
    "rank"
]]

df.to_csv("CAMDA_FINAL_SUBMISSION.csv", index=False)